In [ ]:
import warnings
warnings.filterwarnings('ignore')  # Игнорировать все предупреждения (не рекомендуется в продакшн-коде)
import pandas as pd
import seaborn as sns
from sklearn.linear_model import LogisticRegression, LinearRegression, Lasso
from numpy import mean
from numpy import var
from math import sqrt
import matplotlib.pyplot as plt

TODO  
изучить функционал библиотек  
https://github.com/pyro-ppl/pyro  
https://github.com/py-why/dowhy  
https://github.com/uber/causalml  
Inference and Intervention Causal Models for Business Analysis - чекнуть книгу
  
instrumental variables  
IPTW  
regression discontinuity method  
Bayesian structural time series  
Bayesian Statistics  

**ТЕОРИЯ**  
  
Пусть мы имеем выборку клиентов. Часть этих клиентов подверглась тестовому воздействию, часть - нет.  
Факт вхождения в тестовую группу (0; 1) = **таргет** (например, показ баннера).  
**Целевые метрики** - метрики, которые мы проверяем (повлиял ли на них target - например, выручка)  
**Признаки** - признаки клиентов, которые не зависят от target; но от которых могут зависеть целевые.  
(!) считаем что целевые признаки не должны влиять значимо на факт вхождения в группу  
(типа - всем клиентам с большой выручкой показываем баннер чаще итд).  
В более общей задаче - это просто переменные, определяющие целевую метрику (ковариаты).  
**Конфаундеры** - confounders=сбивающий, признаки, которые влияют <u>и на факт вхождения в группу, и на  
целевые метрики</u>. По сути "третья" переменная которая может скрывать связь целевой метрики и таргета.  
В более общей постановке - это переменная, влияющая одновременно и на X и на Y в задаче поиска связи X-Y.  
Пример: влияют ли спортивные упражнения на состояние здоровья. При этом, вхождение в группу занимающихся  
спортом может зависеть от ожирения -> оно влияет и на разбивку групп и на состояние здоровья.  
Пример2: проверка влияет ли методика с таблетками и с операцией на лечение камней в почках.  
Размер камней не только влияет на излечение, но еще и на фактор отбора в группу A/B (большие -> скорее операция).  

Важно рассматривать полный базис всех признаков, включая признаки-конфаундеры. Тогда будет  
корректной задача по балансированию выборок с симметричным распределением признаков, которые  
оказывают влияние на целевую метрику.

По сути, балансировка заменяет симметричное рандомное сплитование АБ, когда оно возможно.  

Качество балансировки можно проверять, например, через стандартизированные разности:   
Для каждого признака в тестовой/контрольной выборках effect_size_cohen -> 0  
значит что в группах набраны клиенты с примерно одинаковыми признаками.  

Существуют разные способы создавать сбалансированные выборки, одни из них:  
1) simple matching (просто подбираем семплы с target=0;1 и одинаковыми признаками)  
2) PSM (подбираем семплы с target=0;1 у которых близкий PS - шанс попасть в контроль по регрессии)

---
Частный случай несбалансированных по конфаундерам выборок - парадокс Симпсона.  
Смотрим эффект от лекарства в разбивке по молодым и пожилым. Допустим, в обеих группах лекарство помогает.  
Но если объединить пожилых и молодых и их процентный состав разный в A/B -> может произойти инверсия.  
Причина - вылечиваемость зависит не только от лекарства, но и от возраста. Смещение выборок по возрасту ->  
искажение реального эффекта с контролем на возраст.  
https://towardsdatascience.com/solving-simpsons-paradox-e85433c68d03

Возраст влияет не только на излечиваемость, но и на формирование тест/контроль выборки (к примеру,  
более пожилым людям априори дают более проверенное лекарство B, что вносит асимметрию). 

Разрешение парадокса: набирать людей в группы с одинаковым распределением по возрасту    
Это возможно как через рандомное AB так и через пост-балансировку выборок (см PSM).  

---
observational and experimental data.  
В случае с тестовыми данными, которые получены искусственно через АБ тест, мы  
не сталкиваемся с проблемами описанными выше и здесь речь идет скорее о статистической работе над  
чувствительностью детектируемых изменений, валидацией данных и пр.  
Наблюдаемые данные - которые мы собрали (как вариант на прошлой истории) без искусственного сплитования.  
Здесь выборки не сбалансированы, на результат и сплитование может влиять большое кол-во конфаундеров.  
Поэтому необходимо использовать инструменты Causal Inferencе для исследования.


---
Ссылки  
https://www.kaggle.com/code/harrywang/simple-matching-in-python  
https://harrywang.me/psm-did  
Цикл статей о казуальности: https://towardsdatascience.com/why-do-we-need-causality-in-data-science-aec710da021e  
Методы используемые в Uber: https://www.uber.com/en-RS/blog/causal-inference-at-uber/  

Книга по Causal Inference - https://matheusfacure.github.io/python-causality-handbook/landing-page.html

**CACE** = Complier Average Causal Effect  
Пример методики, когда мы проверяем эффект воздействия, которое может быть применено не ко всем в группе.  
Пример: в группе А люди лечатся лекарством 1, при этом достаточно часто его принимают.  
В группе В мы выписываем им новое лекарство - оно сильно более эффективное, но дорогое -> сильно  
реже люди его покупают после рецепта и используют для лечения.  
По итогу, может оказаться, что невыгодно выписывать лекарство В, но если просто сравнивать эффект на  
тех кто его принял - результат будет обратным (конфаундер здесь - цена).  
CACE = (result_B - result_A) / (p_B - p_A), где p - вероятность принять лекарство (изменение).

---
Correlation != Causation  
https://habr.com/ru/companies/ods/articles/544208/  
Причинно следственная связь, однонаправленный вектор: A -> B. Эту схему хорошо задают ацикличные графы.  
Корреляция симметрична, то есть corr(A,B) = corr(B,A) -> теряется информация о направленности.  
Причины, когда корреляция не приводит к каузальности:  
1. Третья переменная. Мы видим A<->B, но в реальности B<-C->A  
(больше церквей -> больше преступности, потому что оба фактора связаны с размером города)
2. Обратная связь. Не A->B, а B->A (курение приводит к депрессии)
3. Selection bias. Делаем выводы только по части выборки вместо всей. 
4. Систематическая ошибка (например, при получении опросов)


#### Difference-in-difference

https://www.kaggle.com/code/harrywang/difference-in-differences-in-python/notebook  
Метод, который используется для оценки влияния изменения на метрику в парадигме до/после.  
Мы ищем параллельный данной метрике тренд (мб из соседнего сегмента) на предыстории -  
и затем в предположении что на параллельный тренд изменение не распространяется, используем  
его в качестве сравнительного контроля (а точнее разницу между трендами до/после)


dif_n_dif = (y_exp_after - y_exp_before) - (y_control_after - y_control_before)  
dif_n_dif -> conf interval -> significance

<u> linear regression: </u>  
y = b0 + b1 * exp_group + b2 * time + b3 * exg_group * time  
exp_group = 1 if exp else 0  
time = 1 if after else 0

y_control_before = b0 ; y_control_after = b0 + b2  
y_exp_before = b0 + b1 ; y_exp_after = b0 + b1 + b2 + b3  
dif_n_dif = (y_exp_after - y_exp_before) - (y_control_after - y_control_before) = b3  

<img src="images/diff_n_diff.png" width="400" align="left">  

In [ ]:
df = pd.read_csv('./data/employment.csv')
df.groupby('state').mean()

In [ ]:
did = (21.096667 - 23.380000) - (20.897249 - 20.430583)
print(did)

In [ ]:
# make total ds
before = df[['state', 'total_emp_feb']]
before['t'] = 0
before.rename(columns={'state' : 'g', 'total_emp_feb' : 'empl_cnt'}, inplace=True)
after = df[['state', 'total_emp_nov']]
after['t'] = 1
after.rename(columns={'state' : 'g', 'total_emp_nov' : 'empl_cnt'}, inplace=True)
ds = pd.concat([before[['g', 't', 'empl_cnt']], after[['g', 't', 'empl_cnt']]], ignore_index=False)
ds['gt'] = ds.g * ds.t

# линейная регрессия
from statsmodels.formula.api import ols
ols = ols('empl_cnt ~ g + t + gt', data=ds).fit()
print(ols.summary())

In [ ]:
# gt = 2.75; p_val = 0.113
# under assumption: parallel trends in con/exp without treatment

In [ ]:
did = []
for _ in range(5000):
    tmp = ds.copy()
    # randomly assign g/t marks (H0 hypothesis)
    tmp['g'] = np.random.permutation(tmp.g.values)
    t1 = tmp[(tmp.t == 0) & (tmp.g == 0)].empl_cnt.mean()
    t2 = tmp[(tmp.t == 0) & (tmp.g == 1)].empl_cnt.mean()
    t3 = tmp[(tmp.t == 1) & (tmp.g == 0)].empl_cnt.mean()
    t4 = tmp[(tmp.t == 1) & (tmp.g == 1)].empl_cnt.mean()
    did_ = (t4 - t3) - (t2 - t1)
    did.append(did_) # h0 hyp, did = 0
# p_value = prob(sample_did extremly than obs_did)
len([j for j in did if abs(j) > abs(obs_did)]) / len(did)

In [ ]:
# p_val совпало с значением в линейной регрессии

In [ ]:
# (!) проверка параллельности трендов в до-тестовый период:
# таким же образом проверяем conf_int_did для t1, t2... < t_exp - должно быть незначительным
# аналогично можно использовать регрессию и показать что фактор gt не играет роли
# https://stats.stackexchange.com/questions/160359/difference-in-difference-method-how-to-test-for-assumption-of-common-trend-betw

#### Matching

**Simple Matching**  
По ограниченному списку признаков для каждого семлпа из одной выборки подбираем зеркальный  
с точным совпадением признаков.

In [ ]:
# видим, что лечение для курильщика вероятно помогает
# однако впрямую control/test нельзя сравнивать так как разбивка не рандомная
df = pd.read_csv('./data/smoker.csv')
df.groupby(['smoker', 'treatment']).outcome.mean().reset_index()

In [ ]:
# 1:1 match - for each person in treatment, we find a match from the control, 
# i.e., if the person is a smoker, we find a smoker in the control
treatment = df[df.treatment == 1]; control = df[df.treatment == 0]

smokers_cnt = min(treatment[treatment.smoker == 1].shape[0], control[control.smoker == 1].shape[0])
non_smokers_cnt = min(treatment[treatment.smoker == 0].shape[0], control[control.smoker == 0].shape[0])

df1 = treatment[treatment.smoker == 1].sample(smokers_cnt, replace=False)
df2 = treatment[treatment.smoker == 0].sample(non_smokers_cnt, replace=False)
df3 = control[control.smoker == 1].sample(smokers_cnt, replace=False)
df4 = control[control.smoker == 0].sample(non_smokers_cnt, replace=False)
df_matched = pd.concat([df1,df2,df3,df4], ignore_index=True)

df.groupby(['treatment', 'smoker']).outcome.mean()

In [ ]:
# распределение показателей не изменилось
df_matched.groupby(['treatment', 'smoker']).outcome.mean()

In [ ]:
# при этом сравнялся состав участников в группах
df_matched.groupby(['treatment', 'smoker']).count()

In [ ]:
# до нормализации создавался ложный эффект, что лечение повышает долю летальных исходов outcome=1
df.groupby('treatment').outcome.mean()

In [ ]:
# после нормализации видим что все окей
df_matched.groupby('treatment').outcome.mean()

**Propensity Score matching**  
Проблемы simple matching в сложных признаковых пространствах - почти нельзя найти  
два семпла с полностью идентичным набором признаков. Поэтому необходимо найти "примерно" похожие.

Идея: используем логистическую регрессию, пробуем оценить вероятность того, что   
семпл принадлежит контролю (например) на базе имеющихся данных. Набираем данные из теста/контроля  
так, чтобы не было перекоса этой вероятности (то есть исходная симметрия).
Propensity score - вероятность принадлежности семпла из выборки контролю.  

---
При этом при оценке регрессией убирают признаки, которые:
1. Прямо коррелируют с фактом вхождения в тестовую группу
2. Связаны с фактом вхождения и могут на него влиять (например, избыточный вес - походы в зал)

Пояснение: если регрессия явно понимает на предложенных признаках где контроль, где тест,  
то признаковое пространство ассиметрично по сути, а значит нельзя сэмулировать похожие семплы  
с/без воздействия (то есть семлпы с/без воздействия всегда будут сильно отличаться)

---
Второй этап - матчинг на основании полученных propensity score (ps).  
Вариант: берем некоторый семпл из теста, и добавляем его в тотал выборку совместно с   
семплом из контроля ps которого максимально близок к семплу из теста (=схожесть семплов).  
Существуют разные способы матчинга используя ps - в том числе когда через kNN мы  
балансируем систему добиваясь минимального расхождения в ps между группами по всей системе итд.

https://harrywang.me/psm-did  
https://towardsdatascience.com/a-hands-on-introduction-to-propensity-score-use-for-beginners-856302b632ac  
https://www.kaggle.com/code/harrywang/propensity-score-matching-in-python/notebook

In [ ]:
df = pd.read_csv('./data/groupon.csv')

In [ ]:
# X = df[['prom_length', 'price', 'discount_pct', 'coupon_duration', 'featured','limited_supply', 'min_req']]
X = df[['prom_length', 'price', 'discount_pct', 'coupon_duration', 'featured','limited_supply']]
y = df['treatment']

lr = LogisticRegression(penalty='l2', C=0.01)
lr.fit(X, y)
pred_prob = lr.predict_proba(X)  
df['ps'] = pred_prob[:, 1]
# Смотрим имеется ли перекрытие - когда модель с равной вероятностью относит к контролю
# и реально контрольные и тестовые семплы (то есть исходно они реально похожи).
# если добавить в признаковое пр-во min_req -> нельзя будет найти похожие семплы в таком пр-ве. 
sns.histplot(data=df, x='ps', hue='treatment')

In [ ]:
# прямой способ матчинга по ближайшим ps - если соседей нет - не берем в расчет
matching = pd.DataFrame(None, columns=['test', 'control']); j=0
df_exp = df[df.treatment == 1][['deal_id', 'ps']].copy()
df_con = df[df.treatment == 0][['deal_id', 'ps']].copy()

for did in df_exp.deal_id.values: # идем по тесту:
    ps = df_exp[(df_exp.deal_id == did)].ps.iloc[0]
    df_con_remain = df_con[~df_con.deal_id.isin(matching.control.values)]
    if len(df_con_remain) == 0:
        pass
    else:
        df_con_remain['dif'] = (df_con_remain['ps'] - ps).apply(abs)
        if df_con_remain.dif.min() > 0.03:
            pass
        else:
            matching.loc[j, :] = did, df_con_remain.sort_values(by='dif', ascending=True).deal_id.iloc[0]
    j+=1

In [ ]:
# валидация результатов через расет стандартизированной разности
def cohen_d(d1, d2):
    n1, n2 = len(d1), len(d2)
    s1, s2 = var(d1, ddof=1), var(d2, ddof=1)
    s = sqrt(((n1 - 1) * s1 + (n2 - 1) * s2) / (n1 + n2 - 2))
    u1, u2 = mean(d1), mean(d2)
    return (u1 - u2) / s

df_con = df[df.deal_id.isin(matching.control.values)]
df_exp = df[df.deal_id.isin(matching.test.values)]
df_con_before = df[df.treatment == 0]
df_exp_before = df[df.treatment == 1]
res_ = []; res_before_ = []

print(matching.shape[0])
# стандартизированная разность до/после балансировки
for col in ['prom_length', 'price', 'discount_pct',
            'coupon_duration', 'featured','limited_supply']:
    res = cohen_d(df_con[col].values, df_exp[col].values)
    res_before = cohen_d(df_con_before[col].values, df_exp_before[col].values)
    res_.append(res); res_before_.append(res_before)
    print(col, '; before', abs(round(res_before, 4)), '; after', abs(round(res, 4)))
print('overall')
print(round(np.mean(res_before_), 4), round(np.mean(res_), 4))

In [ ]:
c1 = df_con.revenue.mean(); e1 = df_exp.revenue.mean() 
c2 = df_con_before.revenue.mean(); e2 = df_exp_before.revenue.mean() 
(e2-c2)/c2, (e1 - c1)/c1

#### Synthetic control

https://matheusfacure.github.io/python-causality-handbook/15-Synthetic-Control.html - основная глава в книге  
https://medium.com/towards-data-science/causal-inference-with-synthetic-control-in-python-4a79ee636325 - доп статья

In [ ]:
# cigar.to_csv('./data/synth_control_smoking.csv') # сохранили на всякий случай

In [ ]:
# cigar = pd.read_stata("https://raw.github.com/scunning1975/mixtape/master/" + 'smoking.dta')
# с 1980х в Калифорнии резко увеличили цену (retail price) за пачку сигарет - стремясь минимизировать курение
cigar[cigar.state == 'California'][['year', 'retprice']].plot(x='year', y='retprice')
cigar['after_treatment'] = cigar.year < 1988
cigar.drop(columns=['beer', 'lnincome', 'age15to24'], inplace=True)
# cigar[cigar.state == 'California'][['year', 'cigsale']].plot(x='year', y='cigsale')
plt.grid()

In [ ]:
plt.plot(cigar[cigar.state == 'California'].year, cigar[cigar.state == 'California'].cigsale)
tmp = cigar[cigar.state != 'California'].groupby('year').cigsale.mean().reset_index()
cigar['state'] = cigar.state.tolist()
plt.plot(tmp.year, tmp.cigsale)
plt.grid()
plt.vlines(x=1988, ymin=40, ymax=140, linestyle=":", lw=2, label="Proposition 99")
plt.title('Динамика продаж на человека до/после закона запрета в Калифорнии')
plt.legend(['california', 'not california_avg'])

По графику мы видим примерно, что в наблюдается ускорение расхождения в потреблении  
в Калифорнии и других штатах после принятия закона, однако нельзя оценить точно.  
Synthetic control - метод построения искуственного контроля (в данном случае штата),   
который ведет себя очень похоже на Калифорнию до изменений - и с большой вероятностью  
моделирует поведение после изменений как если бы их не было.

---
Ключевая идея - мы считаем, что метрику по целевому штату (Калифорния) можно разложить в базис  
вспомогательных штатов (units) на разных временных отрезках. Т.е до внесения изменения,  
можно сделать искуственный штат = комбинация вспомогательных, который будет очень близок Калифорнии.  

Матрица X - по колонкам моменты времени dt, по строкам - штаты (юниты разложения).  
Вектор Y - целевая метрика = продажи в Калифорнии во времени. 
Подбираем веса в тренировке X * w = Y.  
Добиваемся чтобы до интервенции предикт был хороший - используем экстраполяцию на период после.  

PS. Для двух переменных регрессия по векторам x1; x2 ищет коэф. чтобы y = k1 * x1 + k2 * x2  
Каждый вектор x1 = (a1 a2 a3 ... an) - можно рассматривать как набор координат во времени.  
Тогда для каждого "момента времени" y_t = k1 * x1_t + k2 * x2_t и.т.д  
Найдем k1; k2 - можем экстраполировать y_t_next по имеющимся x1_t_next; x2_t_next в будущем.  
Мы через регрессию ищем k1; k2; ... kn для n юнитов разложения (т. е n штатов)  

Важно также помнить о том что при большом кол-ве юнитов (aka предикторов)  
линейная регрессия легко переобучается -> нужна регуляризация.

In [ ]:
dfs = pd.pivot_table(cigar, 
                    index='year', 
                    columns='state', 
                    values=['cigsale'], 
                    aggfunc=np.mean)['cigsale'].reset_index()

df = dfs[dfs.year <= 1988].copy() # train
y = df['California'].values
X = df.drop(columns='California').values

# обучаем регрессию - получаем коэффициенты k1;k2... для каждого штата
# model_ = LinearRegression(fit_intercept=False).fit(X, y) # без регуляризации - переобучение
model_ =  Lasso(fit_intercept=False, alpha=1.0).fit(X, y) # регуляризация по дефолту

# экстраполируем данные на весь период времени
synth = model_.predict(dfs.drop(columns=['California']).values)
real = dfs.California.values
plt.plot(dfs.year.values, real)
plt.plot(dfs.year.values, synth)
plt.grid()
plt.vlines(x=1988, ymin=40, ymax=140, linestyle=":", lw=2, label="Proposition 99")
plt.title('Построение синтетического контроля \n через линейную регрессию с регуляризацией')
plt.legend(['Калифорния real', 'Калифорния synth'])

Проблема регрессии - она экстраполирует, не учитывая "схожесть" регионов друг с другом.  
Например, один из весов модели может быть -100 и уносить показатели одного из штатов в несущ. область.  
Чтобы это избежать - можно находить оптимальные веса оптимизацией (метод взвешенных средних),  
накладывая смысловые ограничения вроде - **сумма всех весов равна 1, веса положительные**.  
Это будет лучше учитывать общую специфику и <u>изначальную схожесть</u> данных в используемом базисе.  
Теперь физический смысл весов - нечто схожее с вероятностью.

In [ ]:
# Реализация метода взвешенных средних через scipy оптимизацию
from typing import List
from operator import add
from toolz import reduce, partial
from scipy.optimize import fmin_slsqp
def loss_w(W, X, y) -> float:
    return np.sqrt(np.mean((y - X.dot(W))**2))
def get_w(X, y):
    
    w_start = [1/X.shape[1]]*X.shape[1]

    weights = fmin_slsqp(partial(loss_w, X=X, y=y),
                         np.array(w_start),
                         f_eqcons=lambda x: np.sum(x) - 1,
                         bounds=[(0.0, 1.0)]*len(w_start),
                         disp=False)
    return weights
calif_weights = get_w(X, y)
# строим
synth = dfs.drop(columns=['California']).values.dot(calif_weights)
real = dfs.California.values
plt.plot(dfs.year.values, real)
plt.plot(dfs.year.values, synth)
plt.grid()
plt.vlines(x=1988, ymin=40, ymax=140, linestyle=":", lw=2, label="Proposition 99")
plt.title('Построение синтетического контроля \n через метод взвешенных средних')
plt.legend(['Калифорния real', 'Калифорния synth'])

In [ ]:
len(calif_synth)